# Summarization and Context Compression

Long support threads blow past context limits. A 40-message ticket can exceed model windows, increase latency and cost, and cause **context rot** — the model attends to noise instead of the latest user need.

**Context compression** keeps the agent effective by shrinking what enters the model **this turn** while preserving facts that matter:

| Technique | What it keeps | What it drops |
|-----------|---------------|---------------|
| **Sliding window** | Last N messages | Oldest turns |
| **Running summary** | Compressed narrative of older turns | Verbatim old chat |
| **Structured scratchpad** | Entities (order_id, name, issue) | Polite filler |
| **Token budget trim** | Highest-priority chunks until budget | Overflow tail |

This notebook implements all four on **ShopCo Support** — plain Python plus a LangGraph compression pipeline — with token estimates, fact-retention evals, and an optional live LLM summarizer.

In [ ]:
"""
ShopCo Support — summarization and context compression lab.
"""

import json
import os
import re
from dataclasses import dataclass, field
from typing import Annotated, Any

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
TOKEN_BUDGET = 120
SLIDING_WINDOW = 6


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)


def message_tokens(messages: list) -> int:
    return sum(estimate_tokens(str(getattr(m, "content", m))) for m in messages)


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
}

print("Compression lab ready.")

---

## 1. The Problem — Growing Context

```
Turn 1 ──► Turn 2 ──► ... ──► Turn 20
   │          │                  │
   └──────────┴──────────────────┴──► ALL in prompt?  ✗ too big
                                      └──► compress ──► model sees summary + recent
```

Checkpointers may store **full** history for audit; the **model context** should be a compressed view.

---

## 2. Synthetic Long Thread for Demos

In [ ]:
def build_long_thread(n_turns: int = 12) -> list:
    """Simulate a lengthy ShopCo support conversation."""
    messages: list = [
        HumanMessage(content="My name is Alice. Order ORD-1001."),
        AIMessage(content="Hi Alice, I see order ORD-1001. How can I help?"),
    ]
    for i in range(2, n_turns):
        if i % 3 == 0:
            messages.append(HumanMessage(content=f"Quick question {i}: what about shipping policy?"))
        elif i % 3 == 1:
            messages.append(AIMessage(content=f"Shipping is free over $50. (reply {i})"))
        else:
            messages.append(HumanMessage(content=f"Follow-up {i}: still worried about my return for ORD-1001."))
    messages.append(HumanMessage(content="So can I return it or not?"))
    return messages


LONG_THREAD = build_long_thread(14)
print(f"Synthetic thread: {len(LONG_THREAD)} messages, ~{message_tokens(LONG_THREAD)} tokens")

---

## 3. Strategy A — Sliding Window

Keep the last `N` messages. Simple, but loses early facts (name, order ID) unless captured elsewhere.

In [ ]:
def sliding_window(messages: list, keep: int = SLIDING_WINDOW) -> list:
    return messages[-keep:] if len(messages) > keep else list(messages)


windowed = sliding_window(LONG_THREAD)
print(f"Window: {len(windowed)} msgs, ~{message_tokens(windowed)} tokens")
print("First kept:", str(windowed[0].content)[:50], "...")

---

## 4. Strategy B — Structured Scratchpad Extraction

Compress conversation into **durable facts** independent of verbatim messages.

In [ ]:
@dataclass
class Scratchpad:
    customer_name: str = ""
    order_id: str = ""
    topics: list[str] = field(default_factory=list)
    open_question: str = ""

    def to_text(self) -> str:
        return (
            f"customer={self.customer_name or '?'}; "
            f"order={self.order_id or '?'}; "
            f"topics={','.join(self.topics) or 'none'}; "
            f"open={self.open_question or 'none'}"
        )


def extract_scratchpad(messages: list) -> Scratchpad:
    pad = Scratchpad()
    for m in messages:
        text = str(m.content)
        if isinstance(m, HumanMessage):
            name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
            if name_m:
                pad.customer_name = name_m.group(1)
            oid = re.search(r"ORD-[0-9]{4}", text.upper())
            if oid:
                pad.order_id = oid.group(0)
            if "return" in text.lower() and "returns" not in pad.topics:
                pad.topics.append("returns")
            if "shipping" in text.lower():
                pad.topics.append("shipping")
            if "?" in text:
                pad.open_question = text.strip()
    return pad


pad = extract_scratchpad(LONG_THREAD)
print("Scratchpad:", pad.to_text(), f"(~{estimate_tokens(pad.to_text())} tokens)")

---

## 5. Strategy C — Running Summary (Offline Summarizer)

Summarize **older** messages into a short narrative; pair with recent verbatim turns.

In [ ]:
def offline_summarize(messages: list, recent_keep: int = 4) -> str:
    """Rule-based summary — stand-in for LLM summarization."""
    if len(messages) <= recent_keep:
        return "(thread too short to summarize)"
    older = messages[:-recent_keep]
    pad = extract_scratchpad(older)
    human_count = sum(1 for m in older if isinstance(m, HumanMessage))
    ai_count = sum(1 for m in older if isinstance(m, AIMessage))
    return (
        f"Earlier thread ({human_count} user, {ai_count} agent msgs): "
        f"{pad.customer_name or 'Customer'} asked about {', '.join(pad.topics) or 'general support'} "
        f"for order {pad.order_id or 'unknown'}."
    )


def live_summarize(messages: list, recent_keep: int = 4) -> str:
    from langchain_openai import ChatOpenAI

    older = messages[:-recent_keep]
    transcript = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Agent'}: {m.content}" for m in older
    )
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content="Summarize in 2 sentences. Keep names, order IDs, and open issues."),
        HumanMessage(content=transcript),
    ])
    return str(resp.content)


def summarize(messages: list) -> str:
    if USE_LIVE_LLM:
        return live_summarize(messages)
    return offline_summarize(messages)


summary = summarize(LONG_THREAD)
print(summary)
print(f"~{estimate_tokens(summary)} tokens")

---

## 6. Strategy D — Token Budget Packing

Pack context until a token budget is reached: summary first, then scratchpad, then recent messages.

In [ ]:
@dataclass
class CompressedContext:
    summary: str
    scratchpad: Scratchpad
    recent_messages: list
    total_tokens: int
    dropped_message_count: int


def compress_thread(
    messages: list,
    *,
    token_budget: int = TOKEN_BUDGET,
    recent_keep: int = 4,
) -> CompressedContext:
    summary = summarize(messages)
    pad = extract_scratchpad(messages)
    recent = messages[-recent_keep:]

    parts = [
        f"SUMMARY: {summary}",
        f"SCRATCHPAD: {pad.to_text()}",
    ]
    used = sum(estimate_tokens(p) for p in parts)

    packed_recent: list = []
    for m in reversed(recent):
        t = estimate_tokens(str(m.content))
        if used + t > token_budget:
            break
        packed_recent.insert(0, m)
        used += t

    return CompressedContext(
        summary=summary,
        scratchpad=pad,
        recent_messages=packed_recent,
        total_tokens=used,
        dropped_message_count=len(messages) - len(packed_recent),
    )


compressed = compress_thread(LONG_THREAD)
print(f"Original: {len(LONG_THREAD)} msgs (~{message_tokens(LONG_THREAD)} tokens)")
print(f"Compressed pack: ~{compressed.total_tokens} tokens, recent={len(compressed.recent_messages)}")
print(f"Dropped from recent window: {compressed.dropped_message_count}")

---

## 7. Build Model Prompt from Compressed Context

In [ ]:
def build_prompt_from_compressed(ctx: CompressedContext) -> list:
    system = SystemMessage(content=(
        "ShopCo support. Use summary + scratchpad + recent messages.\n"
        f"SUMMARY: {ctx.summary}\n"
        f"SCRATCHPAD: {ctx.scratchpad.to_text()}"
    ))
    return [system, *ctx.recent_messages]


def offline_answer(ctx: CompressedContext) -> str:
    last = ctx.recent_messages[-1] if ctx.recent_messages else None
    q = str(last.content).lower() if last else ""
    oid = ctx.scratchpad.order_id
    name = ctx.scratchpad.customer_name or "there"
    if "return" in q:
        return f"Hi {name}, yes — {POLICY_SNIPPETS['returns']} (order {oid})."
    return f"Hi {name}, how can I help with order {oid}?"


prompt = build_prompt_from_compressed(compressed)
print(f"Prompt messages: {len(prompt)}, ~{message_tokens(prompt)} tokens")
print("Answer:", offline_answer(compressed))

---

## 8. LangGraph Compression Pipeline

Graph: **compress** → **respond** — compression is an explicit node, not an afterthought.

In [ ]:
class CompressionState(TypedDict):
    messages: Annotated[list, add_messages]
    running_summary: str
    scratchpad_text: str
    compressed_tokens: int
    answer: str


def compress_node(state: CompressionState) -> dict[str, Any]:
    ctx = compress_thread(state["messages"])
    return {
        "running_summary": ctx.summary,
        "scratchpad_text": ctx.scratchpad.to_text(),
        "compressed_tokens": ctx.total_tokens,
    }


def respond_node(state: CompressionState) -> dict[str, Any]:
    ctx = CompressedContext(
        summary=state.get("running_summary", ""),
        scratchpad=extract_scratchpad(state["messages"]),
        recent_messages=sliding_window(state["messages"], keep=4),
        total_tokens=state.get("compressed_tokens", 0),
        dropped_message_count=0,
    )
    return {"answer": offline_answer(ctx), "messages": [AIMessage(content=offline_answer(ctx))]}


builder = StateGraph(CompressionState)
builder.add_node("compress", compress_node)
builder.add_node("respond", respond_node)
builder.add_edge(START, "compress")
builder.add_edge("compress", "respond")
builder.add_edge("respond", END)

COMPRESSION_GRAPH = builder.compile(checkpointer=MemorySaver())
print("Compression graph compiled.")

---

## 9. Run Compression Graph on Long Thread

In [ ]:
out = COMPRESSION_GRAPH.invoke(
    {"messages": LONG_THREAD, "running_summary": "", "scratchpad_text": "", "compressed_tokens": 0, "answer": ""},
    config={"configurable": {"thread_id": "compress-demo"}},
)

print("Summary:", out["running_summary"])
print("Scratchpad:", out["scratchpad_text"])
print("Compressed tokens:", out["compressed_tokens"])
print("Answer:", out["answer"])

---

## 10. Fact Retention Eval — Compression Must Not Drop Keys

After compression, the agent must still know **Alice**, **ORD-1001**, and **returns**.

In [ ]:
REQUIRED_FACTS = ["Alice", "ORD-1001", "return"]


def fact_retention_score(ctx: CompressedContext, answer: str) -> dict[str, bool]:
    blob = " ".join([ctx.summary, ctx.scratchpad.to_text(), answer]).lower()
    return {fact: fact.lower() in blob for fact in REQUIRED_FACTS}


strategies = {
    "sliding_only": lambda msgs: CompressedContext(
        "", extract_scratchpad(sliding_window(msgs)), sliding_window(msgs), message_tokens(sliding_window(msgs)), 0
    ),
    "full_compress": compress_thread,
}

print(f"{'Strategy':<16} " + " ".join(f"{f:<12}" for f in REQUIRED_FACTS))
print("-" * 55)
for name, fn in strategies.items():
    ctx = fn(LONG_THREAD)
    ans = offline_answer(ctx) if name == "full_compress" else ""
    if name == "sliding_only":
        ans = "return policy?"  # sliding loses name/order — fails eval
    scores = fact_retention_score(ctx, ans)
    marks = ["✓" if scores[f] else "✗" for f in REQUIRED_FACTS]
    print(f"{name:<16} " + " ".join(f"{m:<12}" for m in marks))

---

## 11. When to Summarize vs Trim

| Signal | Action |
|--------|--------|
| `messages` > 20 turns | Summarize older block |
| Token estimate > budget | Token-budget pack |
| Stable entities known | Scratchpad extract + drop old msgs |
| Compliance audit needs full log | Keep full history in checkpointer only |
| Tool outputs huge | Summarize tool result before next model call |

---

## 12. Hierarchical Compression

For very long threads: **turn pairs → episode summary → thread summary**.

In [ ]:
def hierarchical_compress(messages: list, episode_size: int = 4) -> str:
    episodes: list[str] = []
    for i in range(0, len(messages), episode_size):
        chunk = messages[i : i + episode_size]
        pad = extract_scratchpad(chunk)
        episodes.append(f"Ep{i // episode_size + 1}: {pad.to_text()}")
    return " | ".join(episodes)


hier = hierarchical_compress(LONG_THREAD)
print(hier[:200], "...")
print(f"~{estimate_tokens(hier)} tokens vs original ~{message_tokens(LONG_THREAD)}")

---

## 13. Compression + Checkpointer Pattern

```
  Full message list ──► Checkpointer (audit / resume)
         │
         ▼
  compress_node ──► running_summary + scratchpad
         │
         ▼
  respond_node sees: system(summary+scratchpad) + recent msgs
```

Never delete checkpoint history just to save tokens — compress **what the model reads**, not what you store.

---

## 14. Comparison Table

In [ ]:
rows = [
    ("None (full thread)", len(LONG_THREAD), message_tokens(LONG_THREAD), "High", "Best"),
    ("Sliding window", len(sliding_window(LONG_THREAD)), message_tokens(sliding_window(LONG_THREAD)), "Low", "Poor"),
    ("Summary + scratchpad + recent", len(compressed.recent_messages), compressed.total_tokens, "Medium", "Good"),
]

print(f"{'Strategy':<28} {'Msgs in prompt':>14} {'~Tokens':>8} {'Cost':>8} {'Facts':>8}")
print("-" * 70)
for r in rows:
    print(f"{r[0]:<28} {r[1]:>14} {r[2]:>8} {r[3]:>8} {r[4]:>8}")

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Trim without scratchpad | Forgot order ID | Extract entities first |
| Summarize every turn | Cost + latency | Threshold-based compression |
| Delete checkpoint history | No audit trail | Compress prompt only |
| Vague summaries | Lost constraints | Structured scratchpad fields |
| Compress tool errors away | Retry loops break | Keep last error verbatim |

---

## 16. Optional Live LLM Summarization

Set `USE_LIVE_LLM = True` in setup to summarize with `gpt-4o-mini`.

In [ ]:
if USE_LIVE_LLM:
    print(live_summarize(LONG_THREAD))
else:
    print("Offline summary:", summarize(LONG_THREAD))
    print("Set USE_LIVE_LLM = True for LLM-based summarization.")

---

## 17. Quiz

1. Why keep full history in the checkpointer but compress the model prompt?
2. What facts must a ShopCo scratchpad capture?
3. Why does sliding-window-only fail on long threads?
4. When is hierarchical compression useful?
5. What is the difference between `running_summary` and `scratchpad`?

---

## 18. Summary

Context compression keeps agents usable on long ShopCo tickets:

- **Sliding window** — cheap but amnesiac
- **Scratchpad** — structured facts (name, order, topics)
- **Running summary** — narrative of older turns
- **Token budget** — pack summary + scratchpad + recent until limit

Implement compression as an explicit LangGraph node. Measure fact retention — compression that drops `ORD-1001` is worse than no compression at all.